# NB_hospital_daily_summary

Builds `adb_caresyncs.gold.hospital_daily_summary` (one row per hospital per day) from silver lab_results per the STTM *Silver to Gold* v1.0, with hospital and calendar attributes and a day-over-day trend via LAG(). Full rebuild via CREATE OR REPLACE TABLE AS SELECT.

In [0]:
%sql
-- Target : adb_caresyncs.gold.hospital_daily_summary
-- Grain  : one row per hospital per day (driving table silver.lab_results grouped by hospital_id, test_date)
-- Load   : FULL rebuild; LEFT JOINs to hospitals and calendar
-- Trend  : LAG() over hospital_id ordered by summary_date (previous day with activity)
CREATE OR REPLACE TABLE adb_caresyncs.gold.hospital_daily_summary
COMMENT 'Gold: one row per hospital per day with lab test volumes, abnormal/critical rates and day-over-day trend (STTM silver to gold v1.0)'
AS
WITH daily AS (
  SELECT
    hospital_id,
    test_date                                  AS summary_date,
    COUNT(*)                                   AS total_tests,
    COUNT(DISTINCT patient_id)                 AS unique_patients_tested,
    COUNT(DISTINCT doctor_id)                  AS active_doctor_count,
    COUNT_IF(result_status = 'Abnormal')       AS abnormal_test_count,
    COUNT_IF(result_status = 'Critical')       AS critical_test_count
  FROM adb_caresyncs.silver.lab_results
  GROUP BY hospital_id, test_date
),
with_trend AS (
  SELECT
    d.*,
    LAG(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  CAST(t.hospital_id AS INT)                                             AS hospital_id,
  h.hospital_name,
  CAST(t.summary_date AS DATE)                                           AS summary_date,
  c.day_name,
  c.is_weekend,
  CAST(t.total_tests AS INT)                                             AS total_tests,
  CAST(t.unique_patients_tested AS INT)                                  AS unique_patients_tested,
  CAST(t.active_doctor_count AS INT)                                     AS active_doctor_count,
  CAST(t.abnormal_test_count AS INT)                                     AS abnormal_test_count,
  CAST(t.critical_test_count AS INT)                                     AS critical_test_count,
  CAST(t.abnormal_test_count * 100.0 / NULLIF(t.total_tests, 0) AS DECIMAL(5,2)) AS abnormal_rate_pct,
  CAST(t.critical_test_count * 100.0 / NULLIF(t.total_tests, 0) AS DECIMAL(5,2)) AS critical_rate_pct,
  CAST(t.prev_day_total_tests AS INT)                                    AS prev_day_total_tests,
  CAST((t.total_tests - t.prev_day_total_tests) * 100.0 / NULLIF(t.prev_day_total_tests, 0) AS DECIMAL(6,2)) AS test_volume_change_pct
FROM with_trend t
LEFT JOIN adb_caresyncs.silver.hospitals h
  ON t.hospital_id = h.hospital_id
LEFT JOIN adb_caresyncs.silver.calendar c
  ON c.date_key = CAST(date_format(t.summary_date, 'yyyyMMdd') AS INT);

In [0]:
# Return the gold table record count to the caller (e.g. ADF / job task value)
updated_df = spark.read.table("adb_caresyncs.gold.hospital_daily_summary")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))